# DistilBERT Fine-Tuning on AG News (Topic Classification)

Run cells top to bottom. **Runtime > Change runtime type > GPU (T4)** before starting.

If you re-run the install cell after already importing `transformers`, restart the runtime (**Runtime > Restart session**) before continuing, or the newer `eval_strategy` / `processing_class` arguments below may fail.

In [ ]:
!pip install -q -U transformers datasets evaluate accelerate

## Restart runtime here if this is not a fresh session
`Runtime > Restart session`, then continue from the next cell.

In [ ]:
import os
import numpy as np
import torch
import evaluate

from datasets import load_dataset

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(42)

## Environment check

In [ ]:
print("=" * 70)
print("ENVIRONMENT CHECK")
print("=" * 70)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: GPU is not available.")
    print("Training will be significantly slower.")
    print("Go to Runtime > Change runtime type > GPU.")

print("=" * 70)

## Configuration

In [ ]:
MODEL_NAME = "distilbert-base-uncased"

DATASET_NAME = "fancyzhx/ag_news"

MAX_LENGTH = 128

LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01

OUTPUT_DIR = "./topic_results"
FINAL_MODEL_DIR = "./best_distilbert_agnews"

NUM_LABELS = 4

USE_FP16 = torch.cuda.is_available()

LABEL_NAMES = [
    "World",
    "Sports",
    "Business",
    "Sci/Tech"
]

# If you hit an out-of-memory error on a free-tier Colab GPU, lower this
# TRAIN_BATCH_SIZE = 8

## Load AG News dataset

In [ ]:
print("\n" + "=" * 70)
print("LOADING AG NEWS DATASET")
print("=" * 70)

dataset = load_dataset(
    DATASET_NAME
)

print(dataset)

print("\nDataset sizes:")

for split in dataset:
    print(f"{split}: {len(dataset[split])}")

print("\nDataset columns:")
print(dataset["train"].column_names)

## Load tokenizer

In [ ]:
print("\n" + "=" * 70)
print("LOADING TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded successfully.")

## Tokenize dataset

In [ ]:
print("\n" + "=" * 70)
print("TOKENIZING DATASET")
print("=" * 70)


def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )


tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True
)

print("Tokenization completed.")

## Load pretrained DistilBERT

In [ ]:
print("\n" + "=" * 70)
print("LOADING DISTILBERT")
print("=" * 70)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

print("DistilBERT loaded successfully.")

## Evaluation metrics

In [ ]:
print("\n" + "=" * 70)
print("SETTING UP EVALUATION")
print("=" * 70)

accuracy_metric = evaluate.load(
    "accuracy"
)

f1_metric = evaluate.load(
    "f1"
)


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="weighted"
    )

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }


print("Accuracy and F1 metrics ready.")

## Data collator

In [ ]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

## Training configuration

In [ ]:
print("\n" + "=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    num_train_epochs=NUM_EPOCHS,

    weight_decay=WEIGHT_DECAY,

    load_best_model_at_end=True,

    metric_for_best_model="accuracy",
    greater_is_better=True,

    logging_steps=100,

    fp16=USE_FP16,

    save_total_limit=2,

    report_to="none",
)

## Create trainer

In [ ]:
print("\n" + "=" * 70)
print("CREATING TRAINER")
print("=" * 70)

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_datasets["train"],

    eval_dataset=tokenized_datasets["test"],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,
)

print("Trainer ready.")

## Train model

In [ ]:
print("\n" + "=" * 70)
print("STARTING DISTILBERT FINE-TUNING")
print("=" * 70)

trainer.train()

print("\nTraining completed successfully!")

## Final evaluation

In [ ]:
print("\n" + "=" * 70)
print("FINAL MODEL EVALUATION")
print("=" * 70)

evaluation_results = trainer.evaluate()

for key, value in evaluation_results.items():

    if isinstance(value, float):

        print(
            f"{key}: {value:.4f}"
        )

    else:

        print(
            f"{key}: {value}"
        )

## Save best model

In [ ]:
print("\n" + "=" * 70)
print("SAVING FINAL MODEL")
print("=" * 70)

os.makedirs(
    FINAL_MODEL_DIR,
    exist_ok=True
)

trainer.save_model(
    FINAL_MODEL_DIR
)

tokenizer.save_pretrained(
    FINAL_MODEL_DIR
)

print("Model saved successfully!")

print(
    "Model location:",
    os.path.abspath(FINAL_MODEL_DIR)
)

## Verify saved model files

In [ ]:
print("\n" + "=" * 70)
print("SAVED MODEL FILES")
print("=" * 70)

for filename in sorted(
    os.listdir(FINAL_MODEL_DIR)
):

    filepath = os.path.join(
        FINAL_MODEL_DIR,
        filename
    )

    if os.path.isfile(filepath):

        size_mb = (
            os.path.getsize(filepath)
            / (1024 * 1024)
        )

        print(
            f"{filename:<40}"
            f"{size_mb:.2f} MB"
        )

## Reload saved model

In [ ]:
print("\n" + "=" * 70)
print("RELOADING SAVED MODEL")
print("=" * 70)

trained_tokenizer = AutoTokenizer.from_pretrained(
    FINAL_MODEL_DIR
)

trained_model = AutoModelForSequenceClassification.from_pretrained(
    FINAL_MODEL_DIR
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

trained_model.to(device)

trained_model.eval()

print("Saved model successfully reloaded.")
print("Running on:", device)

## Topic classification function

In [ ]:
def predict_topic(text):

    inputs = trained_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = trained_model(
            **inputs
        )

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )

    prediction = torch.argmax(
        probabilities,
        dim=-1
    ).item()

    confidence = probabilities[
        0,
        prediction
    ].item()

    label = LABEL_NAMES[prediction]

    return label, confidence

## Test the fine-tuned model

In [ ]:
print("\n" + "=" * 70)
print("TESTING FINE-TUNED TOPIC CLASSIFIER")
print("=" * 70)


test_sentences = [

    "The president announced a new international agreement.",

    "The football team won the championship after an exciting final.",

    "The company reported record profits this quarter.",

    "Apple announced a new artificial intelligence technology.",

    "Scientists developed a new computer chip for AI applications.",

    "The stock market experienced major changes today.",

    "The national team qualified for the World Cup.",

    "World leaders met to discuss the ongoing conflict."

]


for sentence in test_sentences:

    topic, confidence = predict_topic(
        sentence
    )

    print("\nText:", sentence)

    print(
        "Topic:",
        topic
    )

    print(
        "Confidence:",
        f"{confidence * 100:.2f}%"
    )

## Custom test

In [ ]:
print("\n" + "=" * 70)
print("CUSTOM TEST")
print("=" * 70)

custom_text = (
    "NVIDIA announced a new powerful GPU "
    "designed specifically for artificial intelligence."
)

topic, confidence = predict_topic(
    custom_text
)

print("Text:", custom_text)

print(
    "Predicted Topic:",
    topic
)

print(
    "Confidence:",
    f"{confidence * 100:.2f}%"
)

## Summary

In [ ]:
print("\n" + "=" * 70)
print("TOPIC CLASSIFICATION MODEL COMPLETE")
print("=" * 70)

print("Task: Topic Classification")
print("Base Model:", MODEL_NAME)
print("Dataset: AG News")
print("Number of Classes:", NUM_LABELS)
print("Classes:", LABEL_NAMES)
print("Fine-tuned Model:", FINAL_MODEL_DIR)

print("\nStatus:")
print("Trained       ✓")
print("Evaluated     ✓")
print("Saved         ✓")
print("Reloaded      ✓")
print("Tested        ✓")

print("=" * 70)